# 07 — Visualizations: Global Building Dataset Validation

In [ ]:
ifrom google.colab import drive
import os
if not os.path.ismount('/content/drive'):
    drive.mount('/content/drive')
else:
    print('Drive already mounted.')

In [ ]:
# Install plotting dependencies
import subprocess, sys
# kaleido 0.2.1 uses a bundled renderer — no Chrome needed, reliable in Colab
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "plotly", "kaleido==0.2.1"], check=False)

# Download Barlow font (optional — falls back to Arial if not available)
import os
try:
    os.system("wget -q -O /content/Barlow-Regular.ttf https://github.com/jpt/barlow/raw/main/fonts/ttf/Barlow-Regular.ttf || true")
    os.system("wget -q -O /content/BarlowSemiCondensed-Regular.ttf https://github.com/jpt/barlow/raw/main/fonts/ttf/BarlowSemiCondensed-Regular.ttf || true")
    import matplotlib.font_manager as _fm
    _fm.fontManager.addfont('/content/Barlow-Regular.ttf')
    _fm.fontManager.addfont('/content/BarlowSemiCondensed-Regular.ttf')
except Exception:
    pass

import plotly.graph_objects as go
from plotly.subplots import make_subplots
import pandas as pd
import numpy as np

FONT_FAMILY = 'Barlow, Arial, sans-serif'
FONT_COLOR  = '#1a1a1a'
BG_COLOR    = 'white'

DATASET_COLORS = {
    'overture':               '#1B6CA8',
    'gba':                    '#2BAE82',
    'globfp':                 '#66C7E0',
    'wsf_tracker@10m':        '#C94A35',
    'obt_2023@10m':           '#E0882A',
    'ghsl_built_s_2025@100m': '#CC2266',
    'tempo_2023q4@100m':      '#7B4EBD',
    'wsf_tracker@100m':       '#EFB3A9',
    'obt_2023@100m':          '#F5C98A',
}
DATASET_LABELS = {
    'overture':               'Overture Maps',
    'gba':                    'Global Building Atlas',
    'globfp':                 'GlobFP',
    'obt_2023@10m':           'OBT (10m)',
    'wsf_tracker@10m':        'WSF Tracker (10m)',
    'ghsl_built_s_2025@100m': 'GHSL (100m)',
    'tempo_2023q4@100m':      'TEMPO (100m)',
    'obt_2023@100m':          'OBT (100m)',
    'wsf_tracker@100m':       'WSF Tracker (100m)',
}
SOURCE_COLORS = {
    'SpaceNet7':   '#E05C00',
    'HotOSM':      '#0072B2',
    'Other / Gov': '#009E73',
}

def apply_ppt_style(fig, title=None, height=720, width=1280):
    fig.update_layout(
        font=dict(family=FONT_FAMILY, color=FONT_COLOR, size=15),
        plot_bgcolor=BG_COLOR,
        paper_bgcolor=BG_COLOR,
        height=height,
        width=width,
        title_font=dict(size=20, family=FONT_FAMILY, color=FONT_COLOR),
        title_x=0.5,
        title_xanchor='center',
        legend=dict(
            font=dict(size=13, family=FONT_FAMILY),
            bgcolor='rgba(255,255,255,0.9)',
            bordercolor='#CCCCCC',
            borderwidth=1,
        ),
    )
    fig.update_xaxes(
        title_font=dict(size=15, family=FONT_FAMILY),
        tickfont=dict(size=13, family=FONT_FAMILY),
        gridcolor='#E8E8E8',
    )
    fig.update_yaxes(
        title_font=dict(size=15, family=FONT_FAMILY),
        tickfont=dict(size=13, family=FONT_FAMILY),
        gridcolor='#E8E8E8',
    )
    if title is not None:
        fig.update_layout(title_text=title)
    return fig

print('Setup complete.')


In [ ]:
# ── Only edit this cell ────────────────────────────────────────────────────
DATA_DIR = '/content/drive/MyDrive/WorldBank/FY26 - DEP/Gates Foundation/Building Dataset Validation/outputs/global_metrics/'   # ← set to folder containing your CSVs
# ──────────────────────────────────────────────────────────────────────────
OUT_DIR  = '/content/drive/MyDrive/WorldBank/FY26 - DEP/Gates Foundation/Building Dataset Validation/outputs/figures/'
SENS_DIR = os.path.dirname(DATA_DIR.rstrip('/')) + '/sensitivity_studies/'
os.makedirs(OUT_DIR, exist_ok=True)
print(f'Figures will be saved to: {OUT_DIR}')


In [ ]:
vec = pd.read_csv(DATA_DIR + 'vector_all_cities_merged.csv')
ras = pd.read_csv(DATA_DIR + 'raster_all_cities_merged.csv')

# IoU threshold sensitivity (may not exist yet)
try:
    iou = pd.read_csv(SENS_DIR + 'iou_threshold_sensitivity.csv')
    print(f'iou: {len(iou)} rows')
except FileNotFoundError:
    iou = pd.DataFrame()
    print('iou: file not found — S1/S2 figures will be skipped')

# WSF year-sensitivity outputs
try:
    wsf_sweep   = pd.read_csv(SENS_DIR + 'wsf_year_sensitivity_summary.csv')
    wsf_aligned = pd.read_csv(SENS_DIR + 'wsf_temporal_alignment.csv')
    print(f'wsf_sweep: {len(wsf_sweep)} rows  |  wsf_aligned: {len(wsf_aligned)} rows')
except FileNotFoundError:
    wsf_sweep = pd.DataFrame()
    wsf_aligned = pd.DataFrame()
    print('WSF sensitivity CSVs not found — W1/W2 figures will be skipped')

# Build raster ds_key
ras['ds_key'] = ras['dataset'].astype(str).str.strip().str.lower() + '@' + ras['grid'].astype(str).str.strip().str.lower()

# Reference source classification for vector data
# Reference source classification — read from the AOI tracker (single source of
# truth: the same `source` column whose reference_source drives the IoU threshold
# in vector_runner). No hard-coded city lists, so split/new cities (e.g. *-sn7)
# are picked up automatically.
TRACKER = os.path.dirname(os.path.dirname(DATA_DIR.rstrip('/'))) + '/data/02_interim/aoi_tracker.csv'
_trk = pd.read_csv(TRACKER, dtype=str)
_trk.columns = _trk.columns.str.strip()
_SRC_LABEL = {'spacenet7': 'SpaceNet7', 'hotosm': 'HotOSM'}   # everything else -> Other / Gov
_trk['_city'] = _trk['dataset_folder_name'].astype(str).str.strip()
_trk['_src']  = _trk['source'].astype(str).str.strip().str.lower().map(_SRC_LABEL).fillna('Other / Gov')
_city_source = _trk.drop_duplicates('_city').set_index('_city')['_src']
vec['source'] = vec['city'].astype(str).str.strip().map(_city_source).fillna('Other / Gov')
print('Reference sources (from aoi_tracker.csv):')
print(vec.drop_duplicates('city')['source'].value_counts().to_string())

# Numeric coercion
for _c in ['f1_city','precision_city','recall_city','total_area_bias',
           'count_ratio_total','boundary_f_meanpair_tp']:
    if _c in vec.columns:
        vec[_c] = pd.to_numeric(vec[_c], errors='coerce')

for _c in ['f1_tile_mean','signed_area_bias','rel_area_error_mean']:
    if _c in ras.columns:
        ras[_c] = pd.to_numeric(ras[_c], errors='coerce')

print(f'vec: {len(vec)} rows, {vec["dataset"].nunique()} datasets, {vec["city"].nunique()} cities')
print(f'ras: {len(ras)} rows, {ras["ds_key"].nunique()} ds_keys, {ras["city"].nunique()} cities')
print(f'vec datasets: {sorted(vec["dataset"].unique())}')
print(f'ras ds_keys:  {sorted(ras["ds_key"].unique())}')


In [ ]:
# ── Zero-F1 city exclusion (item 8b) ─────────────────────────────────────────
# Exclude cities where ALL datasets report F1 = 0. These are coverage failures or
# pipeline errors, NOT genuine zero-accuracy results, and they distort global means
# and error bars. Vector and raster are filtered SEPARATELY, each with its own
# criterion. Filtering is in-memory only — the raw CSV files are NOT modified.

# --- Vector: exclude a city if f1_city == 0 for overture AND gba AND globfp ---
VECTOR_FILTER_DATASETS = ['overture', 'gba', 'globfp']
_vpiv = (vec.assign(_ds=vec['dataset'].astype(str).str.strip().str.lower())
            .drop_duplicates(['city', '_ds'])
            .pivot(index='city', columns='_ds', values='f1_city')
            .reindex(columns=VECTOR_FILTER_DATASETS))
# A city is excluded only if all three datasets are present and every one is 0.
# (A missing dataset -> NaN -> NaN == 0 is False -> the city is kept.)
excluded_vec = sorted(_vpiv.index[(_vpiv == 0).all(axis=1)].tolist())
if excluded_vec:
    print(f"[vector] Excluding {len(excluded_vec)} cities (all datasets F1=0): {excluded_vec}")
else:
    print("[vector] No cities excluded (no city has all datasets F1=0).")
vec = vec[~vec['city'].isin(excluded_vec)]
print(f"[vector] Cities remaining: {vec['city'].nunique()}  ({len(vec)} rows)\n")

# --- Raster: exclude a city if f1_tile_mean == 0 for all 6 ds_key values ---
RASTER_FILTER_DSKEYS = [
    'wsf_tracker@10m', 'obt_2023@10m', 'ghsl_built_s_2025@100m',
    'tempo_2023q4@100m', 'wsf_tracker@100m', 'obt_2023@100m',
]
_rpiv = (ras.drop_duplicates(['city', 'ds_key'])
            .pivot(index='city', columns='ds_key', values='f1_tile_mean')
            .reindex(columns=RASTER_FILTER_DSKEYS))
excluded_ras = sorted(_rpiv.index[(_rpiv == 0).all(axis=1)].tolist())
if excluded_ras:
    print(f"[raster] Excluding {len(excluded_ras)} cities (all datasets F1=0): {excluded_ras}")
else:
    print("[raster] No cities excluded (no city has all ds_keys F1=0).")
ras = ras[~ras['city'].isin(excluded_ras)]
print(f"[raster] Cities remaining: {ras['city'].nunique()}  ({len(ras)} rows)")

In [ ]:
VEC_DS = ['overture', 'gba', 'globfp']
_v = vec[vec['dataset'].isin(VEC_DS)].copy()

fig_v1 = go.Figure()

# Helper to slightly darken a hex color for the box outline & median line
def _darken(hex_color, factor=0.6):
    if hex_color.startswith('#') and len(hex_color) == 7:
        return '#' + ''.join(f'{int(int(hex_color[i:i+2], 16) * factor):02x}' for i in (1, 3, 5))
    return '#333333'

for ds in VEC_DS:
    _d = _v[_v['dataset'] == ds]
    ds_label = DATASET_LABELS.get(ds, ds)
    ds_color = DATASET_COLORS.get(ds, '#888')
    ds_color_dark = _darken(ds_color)
    mean_val = _d['f1_city'].mean()

    fig_v1.add_trace(go.Box(
        boxmean=True,
        y=_d['f1_city'],
        name=ds_label,
        line_color=ds_color_dark,   # Darker color for the median line and outline
        fillcolor=ds_color,         # Original dataset color for the fill
        opacity=0.7,
        marker=dict(color='#333333', size=4, opacity=0.8),  # Darker dots to contrast against the box
        showlegend=False,
        boxpoints='all',  # Shows all points alongside the box
        jitter=0.2,       # Spread out dots horizontally
        pointpos=0,       # Center the dots over the box
        text=_d['city'].tolist(),
        hovertemplate='<b>%{text}</b><br>F1: %{y:.3f}<extra></extra>'
    ))

    # Add label for the mean
    fig_v1.add_annotation(
        x=ds_label,
        y=mean_val,
        text=f"<b>Mean: {mean_val:.3f}</b>",
        showarrow=False,
        xshift=75,  # Shift a bit more to accommodate the wider text
        font=dict(size=15, color=FONT_COLOR, family=FONT_FAMILY), # Larger font
        bgcolor='rgba(255,255,255,0.85)',
        bordercolor=ds_color_dark,
        borderwidth=1,
        borderpad=3
    )

apply_ppt_style(fig_v1, title=None, height=500, width=900)  # Title removed
fig_v1.update_layout(yaxis=dict(range=[0, 1.0], title='Macro F1'), xaxis_title='Dataset') # Y-axis capped at 1.0
fig_v1.write_image(os.path.join(OUT_DIR, 'V1_vector_macro_f1.png'), scale=2)
apply_ppt_style(fig_v1)
fig_v1.show()
print('Saved: V1_vector_macro_f1.png')

In [ ]:
import math

_v2 = vec[vec['dataset'].isin(VEC_DS)].copy()
_sources = ['SpaceNet7', 'HotOSM', 'Other / Gov']

fig_v2 = go.Figure()

shown_in_legend = set()

# Map source categories to numerical indices for x-axis positioning
src_to_idx = {src: i for i, src in enumerate(_sources)}

# Define data offsets for each dataset within a group
# These are fractional values relative to the category's center (which is 0.0, 1.0, 2.0 etc.)
# They are determined by the internal spacing of grouped box plots.
# Adjust these values (e.g., +/-0.2, +/-0.25) to fine-tune alignment with box centers.
ds_x_offsets = {
    VEC_DS[0]: -0.3, # Overture Maps (first box in group, shifted left)
    VEC_DS[1]: 0,    # Global Building Atlas (middle box, no shift)
    VEC_DS[2]: 0.3   # GlobFP (third box in group, shifted right)
}

# Define additional correction offsets for each source category
# This adjusts the overall center of the group of annotations if Plotly's default positioning for the category is off.
src_correction_offsets = {
    'SpaceNet7':   -0.08, # Shift entire SpaceNet7 group of annotations slightly left
    'HotOSM':      0,     # No shift for HotOSM
    'Other / Gov': 0.08   # Shift entire Other / Gov group of annotations slightly right
}

for ds in VEC_DS: # Iterate through datasets for grouping and coloring
    _d_all_sources = _v2[_v2['dataset'] == ds]
    ds_label = DATASET_LABELS.get(ds, ds)
    ds_color = DATASET_COLORS.get(ds, '#888') # Get dataset color
    ds_color_dark = _darken(ds_color) # Darken it for lines/borders

    for src in _sources: # Iterate through sources to set x-axis order
        _d_filtered = _d_all_sources[_d_all_sources['source'] == src]

        if not _d_filtered.empty:
            mean_val = _d_filtered['f1_city'].mean() # Calculate mean for this box

            fig_v2.add_trace(go.Box( # Changed from go.Violin to go.Box
                y=_d_filtered['f1_city'],
                x=[src] * len(_d_filtered['f1_city']), # X-axis will be data source
                name=ds_label, # Name will be dataset for grouping and legend
                legendgroup=ds_label, # Group legends by dataset
                showlegend=ds_label not in shown_in_legend, # Show legend only once per dataset
                boxmean=True, # Show mean line inside the box
                line_color=ds_color_dark,   # Darker color for the median line and outline
                fillcolor=ds_color,         # Original dataset color for the fill
                opacity=0.7, # Adjusted opacity for box
                marker=dict(color='#333333', size=4, opacity=0.8),  # Darker dots to contrast against the box
                boxpoints='all',  # Shows all points alongside the box
                jitter=0.2,       # Spread out dots horizontally
                pointpos=0,       # Center the dots over the box
                text=_d_filtered['city'].tolist(),
                hovertemplate='<b>%{text}</b><br>F1: %{y:.3f}<extra></extra>'
            ))
            shown_in_legend.add(ds_label)

            # Add label for the mean, attempting to position it per box
            if not math.isnan(mean_val): # Only add annotation if mean is a valid number
                # Calculate the numerical x-position for the annotation
                # This combines the source category's index with the dataset's fractional offset
                annotation_x_pos = (
                    src_to_idx[src] +
                    ds_x_offsets.get(ds, 0) +
                    src_correction_offsets.get(src, 0)
                )

                fig_v2.add_annotation(
                    x=annotation_x_pos, # Numerical x-position derived from category index + offset
                    y=0.05, # Position within the plot area, near the bottom
                    yref='y', # Reference to the actual y-axis data coordinates
                    text=f"<b>Mean: {mean_val:.3f}</b>",
                    showarrow=False,
                    # xshift is no longer needed as x is precisely positioned in data units
                    yshift=0, # No additional vertical shift
                    font=dict(size=12, color=FONT_COLOR, family=FONT_FAMILY), # Slightly smaller font
                    bgcolor='rgba(255,255,255,0.85)',
                    bordercolor=ds_color_dark,
                    borderwidth=1,
                    borderpad=3,
                    yanchor='bottom' # Anchor the bottom of the text to the y-coordinate
                )

apply_ppt_style(fig_v2,
    title='Vector F1 Distribution by Reference Data Source',
    height=650, width=1100)
fig_v2.update_layout(
    boxmode='group', # Changed from violinmode to boxmode
    boxgap=0.1,      # Reduce whitespace between categories (group of boxes)
    boxgroupgap=0.1, # Reduce whitespace between individual boxes within a group
    yaxis=dict(title='F1 (city level)', range=[0, 1.0]),
    xaxis_title='Reference Data Source',
    xaxis=dict(categoryorder='array', categoryarray=_sources) # Ensure order of sources on x-axis
)
fig_v2.write_image(os.path.join(OUT_DIR, 'V2_f1_by_source.png'), scale=2)
apply_ppt_style(fig_v2)
fig_v2.show()
print('Saved: V2_f1_by_source.png')

In [ ]:
# ── V3: Dataset wins — vector (Donut Chart) ──────────────────────────────────
_v3_cov = vec[vec['dataset'].isin(VEC_DS)].groupby('city')['dataset'].nunique()
_cities_all3 = _v3_cov[_v3_cov == 3].index
vec3 = vec[vec['city'].isin(_cities_all3) & vec['dataset'].isin(VEC_DS)].copy()

# Identify the 'winner' for each city (the dataset with the highest f1_city)
_winner_idx = vec3.groupby('city')['f1_city'].idxmax()
_wins = vec3.loc[_winner_idx, 'dataset'].value_counts()

# Prepare lists for Plotly Pie chart mapping
_labels = [DATASET_LABELS.get(ds, ds) for ds in _wins.index]
_values = _wins.values
_colors = [DATASET_COLORS.get(ds, '#888') for ds in _wins.index]

fig_v3 = go.Figure(data=[go.Pie(
    labels=_labels,
    values=_values,
    hole=0.5,  # Creates the donut shape
    marker=dict(colors=_colors, line=dict(color='white', width=2)),
    texttemplate='<b>%{label}</b><br>%{value} wins (%{percent})',  # Custom text layout
    textfont=dict(size=15, family=FONT_FAMILY),
    hovertemplate='<b>%{label}</b><br>Wins: %{value} (%{percent})<extra></extra>'
)])

apply_ppt_style(fig_v3,
    title=f'Vector Dataset Wins (n={len(_cities_all3)} cities, all datasets present)',
    height=500, width=800)

fig_v3.update_layout(showlegend=False)  # Legend removed as requested
fig_v3.write_image(os.path.join(OUT_DIR, 'V3_vector_wins.png'), scale=2)
apply_ppt_style(fig_v3)
fig_v3.show()
print('Saved: V3_vector_wins.png')


In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import math # Import math for isnan check

# -- V4: Complementary metrics — vector ────────────────────────────────────
_v4 = vec[vec['dataset'].isin(VEC_DS)].copy()
_metrics_v4 = [
    # ('f1_city',              'Macro F1',               'Macro F1 (city level)',       [0, 1],   None, 'mean'),
    ('total_area_bias',  'Total Area Bias',        'Total Area Bias',             [-0.5, 1.5], 0, 'median'),
    ('count_ratio_total',    'Count Ratio',            'Count Ratio (candidate/reference)',    [0, 3],  1.0, 'median'),
]

fig_v4 = make_subplots(rows=1, cols=2,
    subplot_titles=[m[1] for m in _metrics_v4],
    shared_yaxes=True, horizontal_spacing=0.06)

for col_idx, (col, label_title, xaxis_title, x_range, vline, stat_type) in enumerate(_metrics_v4, 1):
    if col not in _v4.columns:
        print(f'[V4] missing column: {col}')
        continue

    for ds in VEC_DS:
        _d = _v4[_v4['dataset'] == ds].dropna(subset=[col])
        ds_label = DATASET_LABELS.get(ds, ds)
        ds_color = DATASET_COLORS.get(ds, '#888')
        ds_color_dark = _darken(ds_color)

        fig_v4.add_trace(go.Box(
            x=_d[col],
            y=[ds_label] * len(_d),
            name=ds_label,
            orientation='h',
            line_color=ds_color_dark,   # Darker color for the median line and outline
            fillcolor=ds_color,         # Original dataset color for the fill
            opacity=0.7, # Adjusted opacity for box
            marker=dict(color='#333333', size=4, opacity=0.8),  # Darker dots to contrast against the box
            boxpoints='all',  # Shows all points alongside the box
            jitter=0.2,       # Spread out dots horizontally
            pointpos=0,       # Center the dots over the box
            showlegend=False,
            boxmean=True,
            text=_d['city'].tolist(), # For hover info
            hovertemplate='<b>%{text}</b><br>' + label_title + ': %{x:.3f}<extra></extra>'
        ), row=1, col=col_idx)

        # Add label for the mean/median
        if stat_type == 'mean':
            val = _d[col].mean()
            text_prefix = 'Mean'
        elif stat_type == 'median':
            val = _d[col].median()
            text_prefix = 'Median'
        else:
            val = float('nan') # Should not happen
            text_prefix = 'Value'

        print(f"DEBUG: {text_prefix} for {ds_label} - {label_title}: {val:.3f}") # Debug print

        if not math.isnan(val): # Only add annotation if value is a valid number
            fig_v4.add_annotation(
                x=val, # X-position is the value on the horizontal axis
                y=ds_label, # Y-position is the categorical dataset label
                text=f"<b>{text_prefix}: {val:.3f}</b>",
                showarrow=False,
                xshift=5,  # Small shift to the right of the line
                font=dict(size=12, color=FONT_COLOR, family=FONT_FAMILY),
                bgcolor='rgba(255,255,255,0.85)',
                bordercolor=ds_color_dark,
                borderwidth=1,
                borderpad=3,
                xanchor='left', # Anchor to the left of the text for positive xshift
                yanchor='middle', # Center vertically on the dataset row
                row=1, col=col_idx # Specify subplot
            )

    if vline is not None:
        fig_v4.add_vline(x=vline, line_dash='dash', line_color='gray', opacity=0.5, row=1, col=col_idx)
    if x_range:
        fig_v4.update_xaxes(range=x_range, row=1, col=col_idx)

    # Explicitly set x-axis title for each subplot
    fig_v4.update_xaxes(title_text=xaxis_title, row=1, col=col_idx)

apply_ppt_style(fig_v4,
    title='Vector Accuracy: F1, Area Bias, and Building Count Ratio',
    height=550, width=1400)
fig_v4.write_image(os.path.join(OUT_DIR, 'V4_complementary_metrics_vector.png'), scale=2)
apply_ppt_style(fig_v4)
fig_v4.show()
print('Saved: V4_complementary_metrics_vector.png')

In [ ]:
# ── V5: F1 vs total area bias scatter ────────────────────────────────────
_v5 = vec[vec['dataset'].isin(VEC_DS)].dropna(subset=['total_area_bias', 'f1_city'])
fig_v5 = go.Figure()
for ds in VEC_DS:
    _d = _v5[_v5['dataset'] == ds]
    fig_v5.add_trace(go.Scatter(
        x=_d['total_area_bias'], y=_d['f1_city'],
        mode='markers', name=DATASET_LABELS.get(ds, ds),
        marker=dict(color=DATASET_COLORS.get(ds, '#888'), size=7, opacity=0.7),
        hovertemplate='<b>%{text}</b><br>Bias: %{x:.3f}<br>F1: %{y:.3f}<extra></extra>',
        text=_d['city'].tolist(),
    ))
fig_v5.add_vline(x=0, line_color='#333', line_width=1.5)
fig_v5.add_vline(x=0.10, line_dash='dash', line_color='gray', opacity=0.5)
fig_v5.add_vline(x=-0.10, line_dash='dash', line_color='gray', opacity=0.5)
apply_ppt_style(fig_v5, title='F1 vs Total Area Bias — Vector Datasets', height=650, width=1000)
fig_v5.update_layout(xaxis_title='Total Area Bias', yaxis_title='F1 (city level)')
fig_v5.write_image(os.path.join(OUT_DIR, 'V5_f1_vs_bias_vector.png'), scale=2)
apply_ppt_style(fig_v5)
fig_v5.show()
print('Saved: V5_f1_vs_bias_vector.png')

# ── V6: Precision vs Recall scatter ───────────────────────────────────────
_v6 = vec[vec['dataset'].isin(VEC_DS)].dropna(subset=['precision_city', 'recall_city'])
fig_v6 = go.Figure()
# F1 iso-curves
for f1_iso in [0.25, 0.50, 0.75]:
    _p = np.linspace(0.01, 1, 200)
    _r = f1_iso * _p / (2 * _p - f1_iso)
    _mask = (_r >= 0) & (_r <= 1)
    fig_v6.add_trace(go.Scatter(
        x=_p[_mask], y=_r[_mask], mode='lines',
        line=dict(color='#AAAAAA', dash='dot', width=1),
        showlegend=False,
        hoverinfo='skip',
    ))
    _mid = len(_p[_mask]) // 2
    if _mid < len(_p[_mask]):
        fig_v6.add_annotation(
            x=float(_p[_mask][_mid]), y=float(_r[_mask][_mid]),
            text=f'F1={f1_iso}', showarrow=False,
            font=dict(size=10, family=FONT_FAMILY, color='#999'),
        )
for ds in VEC_DS:
    _d = _v6[_v6['dataset'] == ds]
    fig_v6.add_trace(go.Scatter(
        x=_d['precision_city'], y=_d['recall_city'],
        mode='markers', name=DATASET_LABELS.get(ds, ds),
        marker=dict(color=DATASET_COLORS.get(ds, '#888'), size=7, opacity=0.7),
        hovertemplate='<b>%{text}</b><br>Prec: %{x:.3f}<br>Rec: %{y:.3f}<extra></extra>',
        text=_d['city'].tolist(),
    ))
apply_ppt_style(fig_v6, title='Precision vs Recall — Vector Datasets (city-level)', height=750, width=900)
fig_v6.update_layout(
    xaxis=dict(title='Precision', range=[0, 1.05]),
    yaxis=dict(title='Recall', range=[0, 1.05]),
)
fig_v6.write_image(os.path.join(OUT_DIR, 'V6_precision_recall.png'), scale=2)
apply_ppt_style(fig_v6)
fig_v6.show()
print('Saved: V6_precision_recall.png')


In [ ]:
# ── V7: Boundary F-score ──────────────────────────────────────────────────
if 'boundary_f_meanpair_tp' not in vec.columns:
    print('[V7] boundary_f_meanpair_tp not in vec — skipping')
else:
    _v7 = vec[vec['dataset'].isin(VEC_DS)].copy()
    _stats7 = (_v7.groupby('dataset')['boundary_f_meanpair_tp']
                .agg(mean='mean', sd='std').reindex(VEC_DS)
                .sort_values('mean', ascending=True).reset_index())
    fig_v7 = go.Figure()
    fig_v7.add_trace(go.Bar(
        y=[DATASET_LABELS.get(d, d) for d in _stats7['dataset']],
        x=_stats7['mean'],
        orientation='h',
        marker_color=[DATASET_COLORS.get(d, '#888') for d in _stats7['dataset']],
        error_x=dict(type='data', array=_stats7['sd'].fillna(0).tolist(), visible=True, thickness=2),
        text=[f'<b>{v:.3f}</b>' for v in _stats7['mean']],
        textposition='outside',
        textfont=dict(size=13, family=FONT_FAMILY, color=FONT_COLOR),
        cliponaxis=False,
    ))
    apply_ppt_style(fig_v7,
        title='Boundary Delineation Accuracy — Vector Datasets',
        height=500, width=900)
    fig_v7.update_layout(
        xaxis=dict(range=[0, 1.15], title='Boundary F-score (mean)'),
        yaxis_title='',
        annotations=[dict(
            x=0.5, y=-0.15, xref='paper', yref='paper', showarrow=False,
            text='Boundary F-score measures delineation accuracy independent of detection.',
            font=dict(size=12, family=FONT_FAMILY, color='#666'),
        )],
    )
    fig_v7.write_image(os.path.join(OUT_DIR, 'V7_boundary_fscore.png'), scale=2)
    apply_ppt_style(fig_v7)
    fig_v7.show()
    print('Saved: V7_boundary_fscore.png')


In [ ]:
# -- V8: Vector macro F1 by IoU-threshold cohort (SpaceNet 0.25 vs other 0.50) --
# The pipeline applies a per-city IoU threshold: 0.25 for SpaceNet reference,
# 0.50 otherwise. V1 pools both cohorts; this splits them so F1 is compared
# like-for-like within each threshold. See notebook 94 for why 0.25 suits the
# SpaceNet7 reference (Planet-imagery boundary uncertainty).
_v8 = vec[vec['dataset'].isin(VEC_DS)].dropna(subset=['f1_city']).copy()
_v8['tau_cohort'] = _v8['iou_threshold'].astype(float).map(
    lambda t: 'SpaceNet (IoU 0.25)' if abs(t - 0.25) < 1e-6 else 'Other (IoU 0.50)')
COHORT_ORDER  = ['SpaceNet (IoU 0.25)', 'Other (IoU 0.50)']
COHORT_COLORS = {'SpaceNet (IoU 0.25)': '#0072B2', 'Other (IoU 0.50)': '#E69F00'}

fig_v8 = go.Figure()
for cohort in COHORT_ORDER:
    _d = _v8[_v8['tau_cohort'] == cohort]
    fig_v8.add_trace(go.Box(
        x=_d['dataset'].map(DATASET_LABELS).fillna(_d['dataset']),
        y=_d['f1_city'],
        name=cohort,
        marker_color=COHORT_COLORS[cohort],
        boxmean=True,
        boxpoints='outliers',
        line=dict(width=1.2),
    ))
apply_ppt_style(fig_v8,
    title='Vector Macro F1 by IoU-threshold cohort (like-for-like within IoU)',
    height=520, width=1000)
fig_v8.update_layout(
    boxmode='group',
    yaxis=dict(title='F1 (city level)', range=[0, 1.05]),
    xaxis_title='Dataset',
    legend=dict(title='Cohort (IoU threshold)'),
)
fig_v8.write_image(os.path.join(OUT_DIR, 'V8_f1_by_tau_cohort.png'), scale=2)
apply_ppt_style(fig_v8)
fig_v8.show()
print('Saved: V8_f1_by_tau_cohort.png')

In [ ]:
# ── R1: Overall macro F1 — raster (Box Plot) ─────────────────────────────────
# RAS_ORDER defines the canonical 6 raster ds_keys (used by R1/R3/R4). It was
# previously set in the removed R1 bar cell; define it here as the first raster cell.
RAS_ORDER = [
    'obt_2023@10m', 'wsf_tracker@10m',
    'ghsl_built_s_2025@100m', 'tempo_2023q4@100m',
    'obt_2023@100m', 'wsf_tracker@100m',
]
_r = ras[ras['ds_key'].isin(RAS_ORDER)].copy()

fig_r1 = go.Figure()

# Helper to slightly darken a hex color for the box outline & median line
def _darken(hex_color, factor=0.6):
    if hex_color.startswith('#') and len(hex_color) == 7:
        return '#' + ''.join(f'{int(int(hex_color[i:i+2], 16) * factor):02x}' for i in (1, 3, 5))
    return '#333333'

for ds in RAS_ORDER:
    _d = _r[_r['ds_key'] == ds]
    ds_label = DATASET_LABELS.get(ds, ds)
    ds_color = DATASET_COLORS.get(ds, '#888')
    ds_color_dark = _darken(ds_color)
    median_val = _d['f1_tile_mean'].median()

    fig_r1.add_trace(go.Box(
        boxmean=True,
        y=_d['f1_tile_mean'],
        name=ds_label,
        line_color=ds_color_dark,   # Darker color for the median line and outline
        fillcolor=ds_color,         # Original dataset color for the fill
        opacity=0.7,
        marker=dict(color='#333333', size=4, opacity=0.8),  # Darker dots to contrast against the box
        showlegend=False,
        boxpoints='all',  # Shows all points alongside the box
        jitter=0.2,       # Spread out dots horizontally
        pointpos=0,       # Center the dots over the box
        text=_d['city'].tolist(),
        hovertemplate='<b>%{text}</b><br>F1: %{y:.3f}<extra></extra>'
    ))

    # Add label for the median
    if not pd.isna(median_val):
        fig_r1.add_annotation(
            x=ds_label,
            y=median_val,
            text=f"<b>Median: {median_val:.3f}</b>",
            showarrow=False,
            xshift=75,  # Shift a bit more to accommodate the wider text
            font=dict(size=14, color=FONT_COLOR, family=FONT_FAMILY),
            bgcolor='rgba(255,255,255,0.85)',
            bordercolor=ds_color_dark,
            borderwidth=1,
            borderpad=3
        )

apply_ppt_style(fig_r1, title=None, height=550, width=1000)  # Title removed, slightly wider for 6 datasets
fig_r1.update_layout(yaxis=dict(range=[0, 1.0], title='Macro F1 (tile mean)'), xaxis_title='Raster Dataset')
fig_r1.write_image(os.path.join(OUT_DIR, 'R1_raster_macro_f1.png'), scale=2)
apply_ppt_style(fig_r1)
fig_r1.show()
print('Saved: R1_raster_macro_f1.png')

In [ ]:
# ── R2: Dataset wins — raster (Donut Charts split by resolution) ─────────────
RAS_10M = ['obt_2023@10m', 'wsf_tracker@10m']
RAS_100M = ['ghsl_built_s_2025@100m', 'tempo_2023q4@100m', 'obt_2023@100m', 'wsf_tracker@100m']

# 10m processing
_cov_10m = ras[ras['ds_key'].isin(RAS_10M)].groupby('city')['ds_key'].nunique()
_cities_10m = _cov_10m[_cov_10m == len(RAS_10M)].index
_r_10m = ras[ras['city'].isin(_cities_10m) & ras['ds_key'].isin(RAS_10M)].copy()
_winner_idx_10m = _r_10m.groupby('city')['f1_tile_mean'].idxmax().dropna()
_winner_10m = _r_10m.loc[_winner_idx_10m]['ds_key'].value_counts()

# 100m processing
_cov_100m = ras[ras['ds_key'].isin(RAS_100M)].groupby('city')['ds_key'].nunique()
_cities_100m = _cov_100m[_cov_100m == len(RAS_100M)].index
_r_100m = ras[ras['city'].isin(_cities_100m) & ras['ds_key'].isin(RAS_100M)].copy()
_winner_idx_100m = _r_100m.groupby('city')['f1_tile_mean'].idxmax().dropna()
_winner_100m = _r_100m.loc[_winner_idx_100m]['ds_key'].value_counts()

# --- Plot 10m ---
_labels_10m = [DATASET_LABELS.get(k, k) for k in _winner_10m.index]
_colors_10m = [DATASET_COLORS.get(k, '#888') for k in _winner_10m.index]

fig_r2_10m = go.Figure(data=[go.Pie(
    labels=_labels_10m, values=_winner_10m.values, hole=0.5,
    marker=dict(colors=_colors_10m, line=dict(color='white', width=2)),
    texttemplate='<b>%{label}</b><br>%{value} wins (%{percent})',
    textfont=dict(size=14, family=FONT_FAMILY),
    hovertemplate='<b>%{label}</b><br>Wins: %{value} (%{percent})<extra></extra>'
)])
apply_ppt_style(fig_r2_10m, title=None, height=450, width=500)
fig_r2_10m.update_layout(showlegend=False)
fig_r2_10m.write_image(os.path.join(OUT_DIR, 'R2_raster_wins_10m.png'), scale=2)
apply_ppt_style(fig_r2_10m)
fig_r2_10m.show()
print('Saved: R2_raster_wins_10m.png')

# --- Plot 100m ---
_labels_100m = [DATASET_LABELS.get(k, k) for k in _winner_100m.index]
_colors_100m = [DATASET_COLORS.get(k, '#888') for k in _winner_100m.index]

fig_r2_100m = go.Figure(data=[go.Pie(
    labels=_labels_100m, values=_winner_100m.values, hole=0.5,
    marker=dict(colors=_colors_100m, line=dict(color='white', width=2)),
    texttemplate='<b>%{label}</b><br>%{value} wins (%{percent})',
    textfont=dict(size=14, family=FONT_FAMILY),
    hovertemplate='<b>%{label}</b><br>Wins: %{value} (%{percent})<extra></extra>'
)])
apply_ppt_style(fig_r2_100m, title=None, height=450, width=500)
fig_r2_100m.update_layout(showlegend=False)
fig_r2_100m.write_image(os.path.join(OUT_DIR, 'R2_raster_wins_100m.png'), scale=2)
apply_ppt_style(fig_r2_100m)
fig_r2_100m.show()
print('Saved: R2_raster_wins_100m.png')


In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import math # Import math for isnan check

# -- R3: Complementary metrics — raster ────────────────────────────────────
_r3 = ras[ras['ds_key'].isin(RAS_ORDER)].copy()
_metrics_r3 = [
    # (col_name, subplot_title, xaxis_title, x_range, vline, stat_type)
    # ('f1_tile_mean',        'Macro F1',          'Macro F1 (tile mean)',        [0, 1],   None, 'mean'),
    # ('signed_area_bias',    'Signed Area Bias',  'Signed Area Bias (median)',     [-0.5, 3], 0, 'median'), # Adjusted range, similar to vector total_area_bias
    ('rel_area_error_mean', 'Rel. Area Error',   'Rel. Area Error (median)',      [-1, 5],  0, 'median'), # Adjusted range, similar to vector count_ratio
]

fig_r3 = make_subplots(rows=1, cols=1,
    subplot_titles=[m[1] for m in _metrics_r3],
    shared_yaxes=True, horizontal_spacing=0.06)

for col_idx, (col, subplot_title, xaxis_title, x_range, vline, stat_type) in enumerate(_metrics_r3, 1):
    if col not in _r3.columns:
        print(f'[R3] missing column: {col}')
        continue

    for ds_key in RAS_ORDER:
        _d = _r3[_r3['ds_key'] == ds_key].dropna(subset=[col])
        ds_label = DATASET_LABELS.get(ds_key, ds_key)
        ds_color = DATASET_COLORS.get(ds_key, '#888')
        ds_color_dark = _darken(ds_color)

        fig_r3.add_trace(go.Box(
            x=_d[col],
            y=[ds_label] * len(_d),
            name=ds_label,
            orientation='h',
            line_color=ds_color_dark,   # Darker color for the median line and outline
            fillcolor=ds_color,         # Original dataset color for the fill
            opacity=0.7, # Adjusted opacity for box
            marker=dict(color='#333333', size=4, opacity=0.8),  # Darker dots to contrast against the box
            boxpoints='all',  # Shows all points alongside the box
            jitter=0.2,       # Spread out dots horizontally
            pointpos=0,       # Center the dots over the box
            showlegend=False,
            boxmean=True,
            text=_d['city'].tolist(), # For hover info
            hovertemplate='<b>%{text}</b><br>' + subplot_title + ': %{x:.3f}<extra></extra>'
        ), row=1, col=col_idx)

        # Add label for the mean/median
        if stat_type == 'mean':
            val = _d[col].mean()
            text_prefix = 'Mean'
        elif stat_type == 'median':
            val = _d[col].median()
            text_prefix = 'Median'
        else:
            val = float('nan') # Should not happen
            text_prefix = 'Value'

        print(f"DEBUG: {text_prefix} for {ds_label} - {subplot_title}: {val:.3f}") # Debug print

        if not math.isnan(val): # Only add annotation if value is a valid number
            fig_r3.add_annotation(
                x=val, # X-position is the value on the horizontal axis
                y=ds_label, # Y-position is the categorical dataset label
                text=f"<b>{text_prefix}: {val:.3f}</b>",
                showarrow=False,
                xshift=5,  # Small shift to the right of the line
                font=dict(size=12, color=FONT_COLOR, family=FONT_FAMILY),
                bgcolor='rgba(255,255,255,0.85)',
                bordercolor=ds_color_dark,
                borderwidth=1,
                borderpad=3,
                xanchor='left', # Anchor to the left of the text for positive xshift
                yanchor='middle', # Center vertically on the dataset row
                row=1, col=col_idx # Specify subplot
            )

    if vline is not None:
        fig_r3.add_vline(x=vline, line_dash='dash', line_color='gray', opacity=0.5, row=1, col=col_idx)
    if x_range: # Update x-axis range only if x_range is defined
        fig_r3.update_xaxes(range=x_range, row=1, col=col_idx)

    # Explicitly set x-axis title for each subplot
    fig_r3.update_xaxes(title_text=xaxis_title, row=1, col=col_idx)

apply_ppt_style(fig_r3,
    title='Raster Accuracy: F1, Area Bias, and Relative Area Error',
    height=550, width=1400)
fig_r3.write_image(os.path.join(OUT_DIR, 'R3_complementary_metrics_raster.png'), scale=2)
apply_ppt_style(fig_r3)
fig_r3.show()
print('Saved: R3_complementary_metrics_raster.png')

In [ ]:
# ── R4: 10m vs 100m resolution comparison ─────────────────────────────────
if 'resolution_m' not in ras.columns:
    ras['resolution_m'] = ras['ds_key'].apply(lambda k: 10 if '10m' in str(k) else 100)

_r4 = ras[ras['ds_key'].isin(RAS_ORDER)].dropna(subset=['f1_tile_mean']).copy()
_r4['res_label'] = _r4['resolution_m'].apply(lambda r: '10m' if r == 10 else '100m')
_res_colors = {'10m': '#C94A35', '100m': '#7B4EBD'}

fig_r4 = go.Figure()
for res_lab in ['10m', '100m']:
    _d = _r4[_r4['res_label'] == res_lab]
    fig_r4.add_trace(go.Violin(
        y=_d['f1_tile_mean'], name=res_lab,
        line=dict(color=_res_colors[res_lab], width=2),
        fillcolor=_res_colors[res_lab], opacity=0.5,
        meanline_visible=True, box_visible=True,
        points='all', pointpos=0, jitter=0.35,
        marker=dict(color=_res_colors[res_lab], size=5, opacity=0.6),
        text=(_d['city'] + ' / ' + _d['ds_key']).tolist(),
        hovertemplate='<b>%{text}</b><br>F1: %{y:.4f}<extra></extra>',
    ))
apply_ppt_style(fig_r4,
    title='Effect of Resolution: 10m vs 100m Raster Datasets (macro F1 per city)',
    height=650, width=800)
fig_r4.update_layout(
    yaxis=dict(title='Macro F1 (tile mean)', range=[-0.05, 1.08]),
    annotations=[dict(
        x=0.5, y=-0.12, xref='paper', yref='paper', showarrow=False,
        text='Higher resolution does not improve F1 — 100m tasks are coarser and easier to match.',
        font=dict(size=12, family=FONT_FAMILY, color='#555'),
    )],
)
fig_r4.write_image(os.path.join(OUT_DIR, 'R4_resolution_comparison.png'), scale=2)
apply_ppt_style(fig_r4)
fig_r4.show()
print('Saved: R4_resolution_comparison.png')


In [ ]:
# -- R5: Raster F1 by reference data source ------------------------------------
import math

_r2 = ras[ras['ds_key'].isin(RAS_ORDER)].copy()

# Map city sources to raster data, similar to how it was done for vector data
# _city_source is already defined in 2927618d from aoi_tracker.csv
_r2['source'] = _r2['city'].astype(str).str.strip().map(_city_source).fillna('Other / Gov')

_sources = ['SpaceNet7', 'HotOSM', 'Other / Gov'] # Keep the same source order

fig_r5 = go.Figure()

shown_in_legend = set()

# Map source categories to numerical indices for x-axis positioning
src_to_idx = {src: i for i, src in enumerate(_sources)}

# Define data offsets for each dataset within a group for 6 raster datasets
# Adjusted to spread 6 boxes within a category. These are approximate and may need fine-tuning.
ds_x_offsets_raster = {
    RAS_ORDER[0]: -0.375, # obt_2023@10m
    RAS_ORDER[1]: -0.225, # wsf_tracker@10m
    RAS_ORDER[2]: -0.075, # ghsl_built_s_2025@100m
    RAS_ORDER[3]: 0.075,  # tempo_2023q4@100m
    RAS_ORDER[4]: 0.225,  # obt_2023@100m
    RAS_ORDER[5]: 0.375   # wsf_tracker@100m
}

# Define additional correction offsets for each source category
# Using the same offsets as the vector plot as they are source-category specific.
src_correction_offsets = {
    'SpaceNet7':   -0.08, # Shift entire SpaceNet7 group of annotations slightly left
    'HotOSM':      0,     # No shift for HotOSM
    'Other / Gov': 0.08   # Shift entire Other / Gov group of annotations slightly right
}

for ds_key in RAS_ORDER: # Iterate through datasets for grouping and coloring
    _d_all_sources = _r2[_r2['ds_key'] == ds_key]
    ds_label = DATASET_LABELS.get(ds_key, ds_key)
    ds_color = DATASET_COLORS.get(ds_key, '#888') # Get dataset color
    ds_color_dark = _darken(ds_color) # Darken it for lines/borders

    for src in _sources: # Iterate through sources to set x-axis order
        _d_filtered = _d_all_sources[_d_all_sources['source'] == src]

        if not _d_filtered.empty:
            mean_val = _d_filtered['f1_tile_mean'].mean() # Calculate mean for this box

            fig_r5.add_trace(go.Box(
                y=_d_filtered['f1_tile_mean'],
                x=[src] * len(_d_filtered['f1_tile_mean']), # X-axis will be data source
                name=ds_label, # Name will be dataset for grouping and legend
                legendgroup=ds_label, # Group legends by dataset
                showlegend=ds_label not in shown_in_legend, # Show legend only once per dataset
                boxmean=True, # Show mean line inside the box
                line_color=ds_color_dark,   # Darker color for the median line and outline
                fillcolor=ds_color,         # Original dataset color for the fill
                opacity=0.7, # Adjusted opacity for box
                marker=dict(color='#333333', size=4, opacity=0.8),  # Darker dots to contrast against the box
                boxpoints='all',  # Shows all points alongside the box
                jitter=0.2,       # Spread out dots horizontally
                pointpos=0,       # Center the dots over the box
                text=_d_filtered['city'].tolist(),
                hovertemplate='<b>%{text}</b><br>F1: %{y:.3f}<extra></extra>'
            ))
            shown_in_legend.add(ds_label)

            # Add label for the mean, attempting to position it per box
            if not math.isnan(mean_val): # Only add annotation if mean is a valid number
                # Calculate the numerical x-position for the annotation
                # This combines the source category's index with the dataset's fractional offset
                annotation_x_pos = (
                    src_to_idx[src] +
                    ds_x_offsets_raster.get(ds_key, 0) + # Use raster specific offsets
                    src_correction_offsets.get(src, 0)
                )

                fig_r5.add_annotation(
                    x=annotation_x_pos, # Numerical x-position derived from category index + offset
                    y=0.05, # Position within the plot area, near the bottom
                    yref='y', # Reference to the actual y-axis data coordinates
                    text=f"<b>{mean_val:.3f}</b>",
                    showarrow=False,
                    yshift=0, # No additional vertical shift
                    font=dict(size=12, color=FONT_COLOR, family=FONT_FAMILY), # Slightly smaller font
                    bgcolor='rgba(255,255,255,0.85)',
                    bordercolor=ds_color_dark,
                    borderwidth=1,
                    borderpad=3,
                    yanchor='bottom' # Anchor the bottom of the text to the y-coordinate
                )

apply_ppt_style(fig_r5,
    title='Raster F1 Distribution by Reference Data Source',
    height=650, width=1100)
fig_r5.update_layout(
    boxmode='group', # Changed from violinmode to boxmode
    boxgap=0.05,      # Reduce whitespace between categories (group of boxes)
    boxgroupgap=0.05, # Reduce whitespace between individual boxes within a group
    yaxis=dict(title='F1 (tile mean)', range=[0, 1.0]),
    xaxis_title='Reference Data Source',
    xaxis=dict(categoryorder='array', categoryarray=_sources) # Ensure order of sources on x-axis
)
fig_r5.write_image(os.path.join(OUT_DIR, 'R5_f1_by_source.png'), scale=2)
apply_ppt_style(fig_r5)
fig_r5.show()
print('Saved: R5_f1_by_source.png')

In [ ]:
# ── S1 & S2: IoU threshold comparison ─────────────────────────────────────
if iou.empty:
    print('[S1/S2] iou DataFrame is empty — skipping')
else:
    # Expected columns: city, dataset, iou_threshold, f1, is_spacenet7
    for _c in ['f1', 'iou_threshold']:
        if _c in iou.columns:
            iou[_c] = pd.to_numeric(iou[_c], errors='coerce')

    _iou_sn7 = iou[iou.get('is_spacenet7', pd.Series(True, index=iou.index)) == True] if 'is_spacenet7' in iou.columns else iou
    _iou_ds = sorted(_iou_sn7['dataset'].dropna().unique()) if 'dataset' in _iou_sn7.columns else []
    _iou_thresholds = sorted(_iou_sn7['iou_threshold'].dropna().unique()) if 'iou_threshold' in _iou_sn7.columns else []

    if len(_iou_thresholds) >= 2 and len(_iou_ds) > 0:
        _t_hi = max(_iou_thresholds)
        _t_lo = min(_iou_thresholds)

        fig_s1 = make_subplots(rows=1, cols=2,
            subplot_titles=[f'By Dataset (SpaceNet7 cities)', f'Pooled across datasets'],
            horizontal_spacing=0.12)

        for ds in _iou_ds:
            for thr, opacity in [(_t_lo, 1.0), (_t_hi, 0.5)]:
                _d = _iou_sn7[(_iou_sn7['dataset'] == ds) & (_iou_sn7['iou_threshold'] == thr)]
                _mean = _d['f1'].mean()
                _sd   = _d['f1'].std()
                fig_s1.add_trace(go.Bar(
                    x=[DATASET_LABELS.get(ds, ds)],
                    y=[_mean],
                    name=f'IoU={thr}',
                    marker_color=DATASET_COLORS.get(ds, '#888'),
                    opacity=opacity,
                    error_y=dict(type='data', array=[_sd], visible=True, thickness=1.5),
                    showlegend=(ds == _iou_ds[0]),
                ), row=1, col=1)

        for thr, opacity in [(_t_lo, 1.0), (_t_hi, 0.5)]:
            _pool = _iou_sn7[_iou_sn7['iou_threshold'] == thr]['f1']
            fig_s1.add_trace(go.Bar(
                x=[f'IoU={thr}'], y=[_pool.mean()],
                name=f'IoU={thr} pooled',
                marker_color='#555' if thr == _t_lo else '#AAA',
                error_y=dict(type='data', array=[_pool.std()], visible=True, thickness=2),
                showlegend=False,
            ), row=1, col=2)

        apply_ppt_style(fig_s1,
            title='Effect of IoU Threshold on F1 Score (SpaceNet7 cities only, macro F1 ± SD)',
            height=600, width=1100)
        fig_s1.update_layout(barmode='group')
        fig_s1.write_image(os.path.join(OUT_DIR, 'S1_iou_threshold.png'), scale=2)
        apply_ppt_style(fig_s1)
        fig_s1.show()
        print('Saved: S1_iou_threshold.png')

        # ── S2: F1 by source — IoU comparison ─────────────────────────────
        if 'source' not in iou.columns and 'city' in iou.columns and 'source' in vec.columns:
            _src_map = vec[['city', 'source']].drop_duplicates('city').set_index('city')['source']
            iou['source'] = iou['city'].map(_src_map).fillna('Other / Gov')

        if 'source' in iou.columns:
            _src_order = ['SpaceNet7', 'HotOSM', 'Other / Gov']
            fig_s2 = go.Figure()
            for thr, opacity in [(_t_lo, 1.0), (_t_hi, 0.55)]:
                _src_stats_s2 = (iou[iou['iou_threshold'] == thr]
                                 .groupby('source')['f1']
                                 .agg(mean='mean', sd='std')
                                 .reindex(_src_order))
                for src in _src_order:
                    if src not in _src_stats_s2.index: continue
                    _mean_s2 = _src_stats_s2.loc[src, 'mean']
                    _sd_s2   = _src_stats_s2.loc[src, 'sd']
                    fig_s2.add_trace(go.Bar(
                        x=[src], y=[_mean_s2],
                        name=f'IoU={thr}',
                        marker_color=SOURCE_COLORS.get(src, '#888'),
                        opacity=opacity,
                        error_y=dict(type='data', array=[_sd_s2], visible=True, thickness=1.5),
                        showlegend=(src == _src_order[0]),
                    ))
            apply_ppt_style(fig_s2,
                title='IoU Threshold Effect by Reference Data Source',
                height=600, width=900)
            fig_s2.update_layout(barmode='group',
                xaxis_title='Reference Data Source', yaxis_title='F1')
            fig_s2.write_image(os.path.join(OUT_DIR, 'S2_iou_by_source.png'), scale=2)
            apply_ppt_style(fig_s2)
            fig_s2.show()
            print('Saved: S2_iou_by_source.png')
    else:
        print(f'[S1/S2] need >= 2 IoU thresholds and datasets; found thresholds={_iou_thresholds}, datasets={len(_iou_ds)}')


In [ ]:
# ── W1: WSF F1 vs reference year ──────────────────────────────────────────
if wsf_sweep.empty:
    print('[W1] wsf_sweep is empty — skipping')
else:
    for _c in ['as_of_year', 'mean_f1', 'mean_precision', 'mean_recall', 'f1_delta_vs_baseline']:
        if _c in wsf_sweep.columns:
            wsf_sweep[_c] = pd.to_numeric(wsf_sweep[_c], errors='coerce')
    wsf_sweep = wsf_sweep.sort_values('as_of_year')

    _baseline_mask = wsf_sweep['f1_delta_vs_baseline'].abs() < 1e-9 if 'f1_delta_vs_baseline' in wsf_sweep.columns else pd.Series(False, index=wsf_sweep.index)
    _baseline_year = float(wsf_sweep.loc[_baseline_mask, 'as_of_year'].iloc[0]) if _baseline_mask.any() else None
    _peak_idx = wsf_sweep['mean_f1'].idxmax()
    _peak_year = float(wsf_sweep.loc[_peak_idx, 'as_of_year'])
    _peak_f1   = float(wsf_sweep.loc[_peak_idx, 'mean_f1'])

    fig_w1 = go.Figure()
    fig_w1.add_trace(go.Scatter(
        x=wsf_sweep['as_of_year'], y=wsf_sweep['mean_f1'],
        mode='lines+markers', name='Mean F1',
        line=dict(color='#C94A35', width=2.5),
        marker=dict(size=7, color='#C94A35'),
    ))
    if 'mean_precision' in wsf_sweep.columns:
        fig_w1.add_trace(go.Scatter(
            x=wsf_sweep['as_of_year'], y=wsf_sweep['mean_precision'],
            mode='lines', name='Mean Precision',
            line=dict(color='#C94A35', width=1.5, dash='dash'), opacity=0.6,
        ))
    if 'mean_recall' in wsf_sweep.columns:
        fig_w1.add_trace(go.Scatter(
            x=wsf_sweep['as_of_year'], y=wsf_sweep['mean_recall'],
            mode='lines', name='Mean Recall',
            line=dict(color='#C94A35', width=1.5, dash='dot'), opacity=0.6,
        ))
    if _baseline_year:
        fig_w1.add_vline(x=_baseline_year, line_dash='dash', line_color='#555', opacity=0.5,
                         annotation_text='baseline', annotation_position='top right')
    fig_w1.add_annotation(
        x=_peak_year, y=_peak_f1,
        text=f'Peak: F1={_peak_f1:.3f} ({_peak_year})',
        arrowhead=2, showarrow=True, arrowcolor='#333',
        font=dict(size=12, family=FONT_FAMILY, color='#333'),
        yshift=12,
    )
    apply_ppt_style(fig_w1,
        title='WSF Tracker Accuracy by Reference Year (year-code sensitivity sweep)',
        height=600, width=1100)
    fig_w1.update_layout(xaxis_title='Reference Year', yaxis_title='Score', yaxis=dict(range=[0, 1.05]))
    fig_w1.write_image(os.path.join(OUT_DIR, 'W1_wsf_year_sensitivity.png'), scale=2)
    apply_ppt_style(fig_w1)
    fig_w1.show()
    print('Saved: W1_wsf_year_sensitivity.png')

# ── W2: Temporal alignment improvement ────────────────────────────────────
if wsf_aligned.empty:
    print('[W2] wsf_aligned is empty — skipping')
else:
    for _c in ['f1_baseline', 'f1_aligned', 'f1_delta']:
        if _c in wsf_aligned.columns:
            wsf_aligned[_c] = pd.to_numeric(wsf_aligned[_c], errors='coerce')
    if 'is_spacenet7' in wsf_aligned.columns:
        wsf_aligned['is_spacenet7'] = wsf_aligned['is_spacenet7'].astype(bool)

    fig_w2 = make_subplots(rows=1, cols=2,
        subplot_titles=['F1: Baseline vs Year-Aligned (per city)', 'Mean F1 Delta by Group'],
        horizontal_spacing=0.12)

    # Left: scatter
    for _is_sn7, _label, _color in [(True, 'SpaceNet7', '#E05C00'), (False, 'Non-SpaceNet7', '#0072B2')]:
        if 'is_spacenet7' in wsf_aligned.columns:
            _d = wsf_aligned[wsf_aligned['is_spacenet7'] == _is_sn7].dropna(subset=['f1_baseline','f1_aligned'])
        else:
            _d = wsf_aligned.dropna(subset=['f1_baseline','f1_aligned'])
            _label = 'All cities'
        if _d.empty: continue
        fig_w2.add_trace(go.Scatter(
            x=_d['f1_baseline'], y=_d['f1_aligned'],
            mode='markers', name=_label,
            marker=dict(color=_color, size=7, opacity=0.75),
            text=_d['city'].tolist() if 'city' in _d.columns else [],
            hovertemplate='<b>%{text}</b><br>Baseline: %{x:.3f}<br>Aligned: %{y:.3f}<extra></extra>',
        ), row=1, col=1)
    # Diagonal y=x
    fig_w2.add_trace(go.Scatter(
        x=[0, 1], y=[0, 1], mode='lines', showlegend=False,
        line=dict(color='gray', width=1, dash='dot'),
    ), row=1, col=1)

    # Right: bar chart of mean delta by group
    _groups_w2 = []
    if 'is_spacenet7' in wsf_aligned.columns and 'f1_delta' in wsf_aligned.columns:
        for _is_sn7, _label in [(True, 'SpaceNet7'), (False, 'Non-SpaceNet7')]:
            _d = wsf_aligned[wsf_aligned['is_spacenet7'] == _is_sn7]['f1_delta'].dropna()
            _groups_w2.append((_label, float(_d.mean()), _d.std()))
        _all_delta = wsf_aligned['f1_delta'].dropna()
        _groups_w2.append(('All', float(_all_delta.mean()), _all_delta.std()))
    elif 'f1_delta' in wsf_aligned.columns:
        _all_delta = wsf_aligned['f1_delta'].dropna()
        _groups_w2 = [('All cities', float(_all_delta.mean()), _all_delta.std())]

    for _label, _mean_d, _sd_d in _groups_w2:
        _color = '#E05C00' if 'Space' in _label else ('#0072B2' if 'Non' in _label else '#555')
        fig_w2.add_trace(go.Bar(
            x=[_label], y=[_mean_d],
            name=_label,
            marker_color=_color,
            error_y=dict(type='data', array=[_sd_d], visible=True, thickness=2),
            text=[f'{_mean_d:+.3f}'],
            textposition='outside',
            textfont=dict(size=13, family=FONT_FAMILY, color=FONT_COLOR),
            showlegend=False,
        ), row=1, col=2)
    fig_w2.add_hline(y=0, line_color='#333', line_width=1.5, row=1, col=2)

    apply_ppt_style(fig_w2,
        title='Per-City Temporal Alignment Improves WSF Tracker Accuracy',
        height=600, width=1200)
    fig_w2.update_xaxes(title_text='F1 (pipeline baseline)', row=1, col=1)
    fig_w2.update_yaxes(title_text='F1 (year-aligned to reference)', row=1, col=1)
    fig_w2.update_yaxes(title_text='Mean F1 Delta', row=1, col=2)
    fig_w2.write_image(os.path.join(OUT_DIR, 'W2_wsf_alignment.png'), scale=2)
    apply_ppt_style(fig_w2)
    fig_w2.show()
    print('Saved: W2_wsf_alignment.png')


In [ ]:
# -- D1 / D2: Per-tile F1 vs reference building DENSITY and SIZE (vector) -------
# Tile-level enriched table (one row per city x dataset x tile). F1 is at each
# city's IoU threshold, so read the TREND across quartiles, not absolute levels.
import numpy as np
_TILE_CSV = os.path.dirname(DATA_DIR.rstrip('/')) + '/scratch/per_tile_enriched_all_cities.csv'

def _q4(series):
    """Quartile categorical with value-range labels. Rank-based, so it never
    fails on duplicate bin edges."""
    q = series.quantile([0, .25, .5, .75, 1]).values
    labels = [f'Q1\n(<{q[1]:.0f})', f'Q2\n({q[1]:.0f}-{q[2]:.0f})',
              f'Q3\n({q[2]:.0f}-{q[3]:.0f})', f'Q4\n(>{q[3]:.0f})']
    return pd.qcut(series.rank(method='first'), 4, labels=labels)

def _f1_box_by_quartile(df, qcol, group_col, group_order, title, xlabel, fname, width=1000):
    """Grouped box: quartile on X (low -> high), one colour per dataset.
    Reading left -> right shows the density/size effect on F1."""
    order = list(df[qcol].cat.categories)
    fig = go.Figure()
    for g in group_order:
        _d = df[df[group_col] == g]
        if _d.empty:
            continue
        fig.add_trace(go.Box(
            x=_d[qcol].astype(str), y=_d['f1'],
            name=DATASET_LABELS.get(g, g),
            marker_color=DATASET_COLORS.get(g, '#888'),
            boxmean=True, boxpoints=False, line=dict(width=1.2),
        ))
    apply_ppt_style(fig, title=title, height=520, width=width)
    fig.update_layout(boxmode='group',
        xaxis=dict(title=xlabel, categoryorder='array', categoryarray=order),
        yaxis=dict(title='Tile F1', range=[0, 1.05]), legend=dict(title='Dataset'))
    fig.write_image(os.path.join(OUT_DIR, fname), scale=2)
    apply_ppt_style(fig); fig.show()
    print('Saved:', fname, f'({len(df):,} tiles)')

tile = pd.read_csv(_TILE_CSV)
tile['dataset'] = tile['dataset'].astype(str).str.lower()
tile = tile[tile['dataset'].isin(VEC_DS)].copy()
for _c in ['f1', 'ref_building_density_per_km2', 'mean_ref_building_area_m2', 'ref_building_count_centroid']:
    tile[_c] = pd.to_numeric(tile[_c], errors='coerce')
tile = tile.dropna(subset=['f1', 'ref_building_density_per_km2', 'mean_ref_building_area_m2'])
tile = tile[(tile['ref_building_count_centroid'] > 0) &
            (tile['ref_building_density_per_km2'] > 0) &
            (tile['mean_ref_building_area_m2'] > 0)]
tile['density_q'] = _q4(tile['ref_building_density_per_km2'])
tile['size_q']    = _q4(tile['mean_ref_building_area_m2'])
print(f'Vector density/size: {len(tile):,} tiles ({tile["city"].nunique()} cities)')

_f1_box_by_quartile(tile, 'density_q', 'dataset', VEC_DS,
    'Per-tile F1 by reference building density (vector)',
    'Reference building density quartile (bldg/km2, low -> high)', 'D1_f1_by_density.png')
_f1_box_by_quartile(tile, 'size_q', 'dataset', VEC_DS,
    'Per-tile F1 by reference building size (vector)',
    'Mean reference building size quartile (m2, small -> large)', 'D2_f1_by_size.png')

In [ ]:
# -- D3 / D4: Per-tile F1 vs reference building DENSITY and SIZE (raster) -------
# Raster per-tile F1 lives in the per-city raster_metrics_tiles_all_datasets.parquet
# files (NOT per_tile_enriched, which is vector-only). Load those, then join tile
# density/size from per_tile_enriched by (city, tile_id) -- density is a property
# of the reference buildings in a tile, shared across candidate datasets.
from pathlib import Path
_METRICS = Path(os.path.dirname(DATA_DIR.rstrip('/'))) / 'metrics'

_dens = pd.read_csv(_TILE_CSV, usecols=['city', 'tile_id', 'ref_building_density_per_km2',
                                        'mean_ref_building_area_m2', 'ref_building_count_centroid'])
_dens = _dens.drop_duplicates(['city', 'tile_id'])

_parts = []
for _p in sorted(_METRICS.rglob('raster_metrics_tiles_all_datasets.parquet')):
    try:
        _parts.append(pd.read_parquet(_p, columns=['city', 'tile_id', 'dataset', 'grid', 'f1']))
    except Exception as _e:
        print(f'  [skip] {_p.parent.name}: {_e}')
rtile = pd.concat(_parts, ignore_index=True) if _parts else pd.DataFrame()
print(f'Raster tiles loaded: {len(rtile):,} from {len(_parts)} cities')

if rtile.empty:
    print('[D3/D4] no raster tile parquets found under outputs/metrics/ -- run 02/03 first.')
else:
    rtile['ds_key'] = (rtile['dataset'].astype(str).str.strip().str.lower() + '@' +
                       rtile['grid'].astype(str).str.strip().str.lower())
    rtile = rtile[rtile['ds_key'].isin(RAS_ORDER)].merge(_dens, on=['city', 'tile_id'], how='left')
    for _c in ['f1', 'ref_building_density_per_km2', 'mean_ref_building_area_m2', 'ref_building_count_centroid']:
        rtile[_c] = pd.to_numeric(rtile[_c], errors='coerce')
    rtile = rtile.dropna(subset=['f1', 'ref_building_density_per_km2', 'mean_ref_building_area_m2'])
    rtile = rtile[(rtile['ref_building_count_centroid'] > 0) &
                  (rtile['ref_building_density_per_km2'] > 0) &
                  (rtile['mean_ref_building_area_m2'] > 0)]
    print(f'Raster density/size: {len(rtile):,} tiles ({rtile["city"].nunique()} cities)')

    if rtile.empty:
        print('[D3/D4] no tiles after density join -- check tile_id alignment.')
    else:
        rtile['density_q'] = _q4(rtile['ref_building_density_per_km2'])
        rtile['size_q']    = _q4(rtile['mean_ref_building_area_m2'])
        _RAS_PRESENT = [k for k in RAS_ORDER if k in set(rtile['ds_key'])]
        _f1_box_by_quartile(rtile, 'density_q', 'ds_key', _RAS_PRESENT,
            'Per-tile F1 by reference building density (raster)',
            'Reference building density quartile (bldg/km2, low -> high)',
            'D3_f1_by_density_raster.png', width=1300)
        _f1_box_by_quartile(rtile, 'size_q', 'ds_key', _RAS_PRESENT,
            'Per-tile F1 by reference building size (raster)',
            'Mean reference building size quartile (m2, small -> large)',
            'D4_f1_by_size_raster.png', width=1300)

In [ ]:
# -- Summary of saved figures (updated) -----------------------------------------
# Re-define _figures to include the new raster plots
_figures = [
    ('V1_vector_macro_f1.png',          'Vector macro F1 per dataset (box; mean + median; pooled)'),
    ('V2_f1_by_source.png',             'Vector F1 broken out by reference data source'),
    ('V3_vector_wins.png',              'Donut: which vector dataset ranks best per city'),
    ('V4_complementary_metrics_vector.png', 'Vector: F1, total area bias, count ratio'),
    ('V5_f1_vs_bias_vector.png',        'Scatter: F1 vs total area bias (city level)'),
    ('V6_precision_recall.png',         'Scatter: precision vs recall with F1 iso-curves'),
    ('V7_boundary_fscore.png',          'Boundary F-score per vector dataset'),
    ('V8_f1_by_tau_cohort.png',         'Vector macro F1 split by IoU cohort (0.25 vs 0.50)'),
    ('R1_raster_macro_f1.png',          'Raster macro F1 per ds_key (box; mean + median)'),
    ('R2_raster_wins_10m.png',          'Donut: best 10m raster dataset per city'),
    ('R2_raster_wins_100m.png',         'Donut: best 100m raster dataset per city'),
    ('R3_complementary_metrics_raster.png', 'Raster: F1, signed area bias, rel. area error'),
    ('R4_resolution_comparison.png',    'Violin: 10m vs 100m raster F1 distribution'),
    ('R5_f1_by_source.png',             'Raster F1 distribution by reference data source'),
    ('S1_iou_threshold.png',            'IoU threshold effect on F1 (SpaceNet)'),
    ('S2_iou_by_source.png',            'IoU threshold effect by reference data source'),
    ('W1_wsf_year_sensitivity.png',     'WSF F1 vs reference year sweep'),
    ('W2_wsf_alignment.png',            'WSF temporal alignment: scatter + delta bar'),
    ('D1_f1_by_density.png',            'Per-tile F1 by reference building density quartile (vector)'),
    ('D2_f1_by_size.png',               'Per-tile F1 by reference building size quartile (vector)'),
    ('D3_f1_by_density_raster.png',     'Per-tile F1 by reference building density quartile (raster)'),
    ('D4_f1_by_size_raster.png',        'Per-tile F1 by reference building size quartile (raster)')
]

print(f'{"Figure":<45} Description')
print('-' * 100)
for fname, desc in _figures:
    _exists = '[x]' if os.path.exists(os.path.join(OUT_DIR, fname)) else '[ ]'
    print(f'{_exists}  {fname:<43} {desc}')
print(f'\nFigures saved to: {OUT_DIR}')